## 1. Start spark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("olist-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.postgresql:postgresql:42.7.3"
    )
    .getOrCreate()
)

spark

# 2. Configurasi

In [3]:
import os

DATA_PATH = os.getenv('DATA_PATH')
RAW_PATH = os.getenv('RAW_PATH')

print('DATA_PATH =', DATA_PATH)
print('RAW_PATH =', RAW_PATH)

DATA_PATH = /home/jovyan/work/data
RAW_PATH = /home/jovyan/work/data/raw


In [4]:
spark.version

'3.5.0'

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()  # pastikan ini dipanggil sebelum os.getenv

RAW_PATH = os.getenv("RAW_PATH")
print(RAW_PATH)  # cek apakah terbaca atau None

/home/jovyan/work/data/raw


In [6]:
# DATA_PATH is now loaded from environment (see config cell above)

df = spark.read.csv(f"{RAW_PATH}/orders/olist_orders_dataset.csv", header=True, inferSchema=True)

In [7]:
df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [8]:
df.count()

99441

# 3. Test Spark to PostgreSQL Connection

In [11]:
jdbc_url = "jdbc:postgresql://postgres:5432/olist_dw"

connection_properties = {
    "user": "airflow",
    "password": "airflow",
    "driver": "org.postgresql.Driver"
}

print("Koneksi berhasil!")

Koneksi berhasil!


In [12]:
test_df = spark.createDataFrame(
    [
        (1, "spark_ok"),
        (2, "postgres_ok")
    ],
    ["id", "status"]
)

test_df.show()

+---+-----------+
| id|     status|
+---+-----------+
|  1|   spark_ok|
|  2|postgres_ok|
+---+-----------+



In [14]:
test_df.write \
    .mode("overwrite") \
    .jdbc(
        url=jdbc_url,
        table="staging.spark_connection_test",
        properties=connection_properties
    )

In [15]:
read_back_df = spark.read.jdbc(
    url=jdbc_url,
    table="staging.spark_connection_test",
    properties=connection_properties
)

read_back_df.show()

+---+-----------+
| id|     status|
+---+-----------+
|  1|   spark_ok|
|  2|postgres_ok|
+---+-----------+



In [17]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="olist_dw",
    user="airflow",
    password="airflow"
)
conn.autocommit = True

cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS staging;")
cur.close()
conn.close()

print("staging schema ready")

staging schema ready


# 4. Read Raw Orders

In [19]:
orders_df = spark.read.csv(
    f"{RAW_PATH}/orders/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

orders_df.printSchema()
orders_df.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

## 5. Selesct and Cast Columns

In [20]:
from pyspark.sql import functions as F

orders_clean_df = (
    orders_df
    .select(
        F.col("order_id").cast("string"),
        F.col("customer_id").cast("string"),
        F.col("order_status").cast("string"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
        F.to_timestamp("order_approved_at").alias("order_approved_at"),
        F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_date"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_date"),
    )
)

## 6. Basic quality filter

In [21]:
orders_valid_df = (
    orders_clean_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["order_id"])
)

orders_valid_df.count()

99441

In [22]:
orders_valid_df.show(5, truncate=False)
orders_valid_df.printSchema()

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|00018f77f2f0320c557190d7a144bdd3|f6dd3ec061db4e3987629fe6b26e5cce|delivered   |2017-04-26 10:53:06     |2017-04-26 11:05:13|2017-05-04 14:35:00         |2017-05-12 16:04:24          |2017-05-15 00:00:00          |
|00042b26cf59d7ce69dfabb4e55b4fd9|58dbd0b2d70206bf40e62cd34e84d795|delivered   |2017-02-04 13:57:51     |2017-02-04 14:10:13|2017-02-16 09:4

# 7. Create Target Table if Needed

In [24]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="olist_dw",
    user="airflow",
    password="airflow"
)
conn.autocommit = True

cur = conn.cursor()
cur.execute("""
CREATE SCHEMA IF NOT EXISTS staging;

CREATE TABLE IF NOT EXISTS staging.orders_stream (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
cur.close()
conn.close()

print("staging.orders_stream ready")

staging.orders_stream ready


In [25]:
orders_valid_df.write \
    .mode("overwrite") \
    .jdbc(
        url=jdbc_url,
        table="staging.orders_stream",
        properties=connection_properties
    )

In [26]:
orders_from_pg_df = spark.read.jdbc(
    url=jdbc_url,
    table="staging.orders_stream",
    properties=connection_properties
)

orders_from_pg_df.show(5, truncate=False)
orders_from_pg_df.count()

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|0006ec9db01a64e59a68b2c340bf65a7|5d178120c29c61748ea95bac23cb8f25|delivered   |2018-07-24 17:04:17     |2018-07-24 17:24:20|2018-07-25 11:02:00         |2018-07-31 01:04:15          |2018-08-22 00:00:00          |
|0015ebb40fb17286bea51d4607c4733c|da43a556bf5c36a1104c473cff77de6c|delivered   |2018-01-14 09:01:36     |2018-01-14 09:11:24|2018-01-17 22:5

99441

In [27]:
print("raw orders count:", orders_df.count())
print("valid deduplicated orders count:", orders_valid_df.count())
print("postgres orders count:", orders_from_pg_df.count())

raw orders count: 99441
valid deduplicated orders count: 99441
postgres orders count: 99441
